# 02. Causal Treatment Effects: TARNet, CFRNet, Dragonnet, Double Machine Learning (DML), and R-Learner

Estimating heterogeneous treatment effects from observational data is a core capability required for precision medicine, dynamic pricing, and scientific experimentation. Unlike conventional supervised learning, which predicts observed outcomes $Y$, causal inference requires estimating counterfactual outcomes $Y(1)$ and $Y(0)$ where only one potential outcome is observed per individual.

### Core Causal Pillars
1. **TARNet Dual Outcome Heads**: Routes representations through a shared trunk $\Phi(x)$ into independent control ($Y_0$) and treatment ($Y_1$) prediction heads to estimate CATE: $\tau(x) = Y_1 - Y_0$.
2. **CFRNet Representation Balancing**: Adds an Integral Probability Metric (MMD) penalty between treated and control latent distributions to prevent covariate extrapolation.
3. **Dragonnet Propensity Sufficiency**: Integrates a propensity score head $\hat{e}(x)$ directly into the shared trunk to preserve confounding features.
4. **Double Machine Learning (DML)**: Employs the Robinson orthogonal transformation to ensure root-$n$ consistent treatment effect estimation even when nuisance models converge at slower rates.


## Mathematical Foundations & Governing Equations

### 1. TARNet Potential Outcome Architecture
$$\Phi(x) = \text{ReLU}(x \Phi_W + \Phi_b)$$
$$Y_0 = \Phi(x) h_{0, W} + h_{0, b}, \quad Y_1 = \Phi(x) h_{1, W} + h_{1, b}, \quad \tau(x) = Y_1 - Y_0$$

### 2. CFRNet Maximum Mean Discrepancy (MMD) Penalty
$$\mu_1 = \frac{1}{N_1} \sum_{i: T_i = 1} \Phi(x_i), \quad \mu_0 = \frac{1}{N_0} \sum_{i: T_i = 0} \Phi(x_i), \quad \text{MMD}^2 = \|\mu_1 - \mu_0\|_2^2$$

### 3. Dragonnet Propensity Loss
$$\hat{e}(x) = \sigma(\Phi(x) w_{\text{prop}} + b_{\text{prop}}), \quad \mathcal{L}_{\text{BCE}} = -\frac{1}{N} \sum_{i=1}^N \left[ T_i \ln \hat{e}(x_i) + (1 - T_i) \ln(1 - \hat{e}(x_i)) \right]$$

### 4. Double Machine Learning (DML) Robinson Estimator
$$\tilde{Y}_i = Y_i - \hat{m}(X_i), \quad \tilde{T}_i = T_i - \hat{e}(X_i)$$
$$\hat{\tau}_{\text{DML}} = \frac{\sum_{i=1}^N \tilde{T}_i \tilde{Y}_i}{\sum_{i=1}^N \tilde{T}_i^2}$$


In [ ]:
# Setup, imports, and golden weight loading
import sys
from pathlib import Path
import torch
import torch.nn.functional as F
from safetensors.torch import load_file

repo_root = Path.cwd()
while not (repo_root / "Cargo.toml").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

weights_path = repo_root / "weights" / "09_causal_ai_and_evolutionary_search" / "02_causal_treatment_effects_tarnet_cfrnet_dragonnet_dml_and_r_learner.safetensors"
assert weights_path.exists(), f"Weights file not found: {weights_path}"
weights = load_file(str(weights_path))
print(f"Loaded {len(weights)} golden tensors on cpu")


In [ ]:
def linear_anchor(x: torch.Tensor, weight: torch.Tensor, bias: torch.Tensor) -> torch.Tensor:
    return x @ weight + bias

def tarnet_forward(x, phi_w, phi_b, h0_w, h0_b, h1_w, h1_b):
    phi = F.relu(x @ phi_w + phi_b)
    y0 = phi @ h0_w + h0_b
    y1 = phi @ h1_w + h1_b
    return phi, y0, y1, y1 - y0

def cfrnet_mmd_loss(phi: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
    t_mask = t.squeeze(-1)
    n_t, n_c = t_mask.sum().clamp(min=1.0), (1.0 - t_mask).sum().clamp(min=1.0)
    mu_1 = (phi * t_mask.unsqueeze(-1)).sum(dim=0) / n_t
    mu_0 = (phi * (1.0 - t_mask).unsqueeze(-1)).sum(dim=0) / n_c
    return torch.sum((mu_1 - mu_0) ** 2).unsqueeze(0)

def dragonnet_propensity_loss(phi, t, w_prop, b_prop, eps=1e-8):
    propensity = torch.sigmoid(phi @ w_prop + b_prop)
    p_c = propensity.clamp(min=eps, max=1.0 - eps)
    bce = -torch.mean(t * torch.log(p_c) + (1.0 - t) * torch.log(1.0 - p_c)).unsqueeze(0)
    return propensity, bce

def dml_cate_estimate(y, t, m_hat, e_hat):
    res_t, res_y = t - e_hat, y - m_hat
    return (torch.sum(res_t * res_y) / torch.sum(res_t * res_t).clamp(min=1e-8)).unsqueeze(0)


In [ ]:
# Multi-track numerical parity assertions
y_anchor = linear_anchor(weights["test_input"], weights["weight"], weights["bias"])
err_anchor = float(torch.max(torch.abs(y_anchor - weights["test_output"])))
assert err_anchor < 1e-5, f"Anchor parity error: {err_anchor:.2e}"

phi, y0, y1, tau = tarnet_forward(weights["test_input"], weights["phi_w"], weights["phi_b"], weights["h0_w"], weights["h0_b"], weights["h1_w"], weights["h1_b"])
err_y0 = float(torch.max(torch.abs(y0 - weights["y0_ref"])))
err_y1 = float(torch.max(torch.abs(y1 - weights["y1_ref"])))
err_tau = float(torch.max(torch.abs(tau - weights["tau_ref"])))
assert max(err_y0, err_y1, err_tau) < 1e-5, f"TARNet parity error"

mmd = cfrnet_mmd_loss(phi, weights["t_treatment"])
err_mmd = float(torch.max(torch.abs(mmd - weights["mmd_loss_ref"])))
assert err_mmd < 1e-5, f"CFRNet MMD parity error: {err_mmd:.2e}"

prop, bce = dragonnet_propensity_loss(phi, weights["t_treatment"], weights["w_prop"], weights["b_prop"])
err_prop = float(torch.max(torch.abs(prop - weights["propensity_ref"])))
err_bce = float(torch.max(torch.abs(bce - weights["prop_loss_ref"])))
assert max(err_prop, err_bce) < 1e-5, f"Dragonnet parity error"

dml_tau = dml_cate_estimate(weights["y_outcome"], weights["t_treatment"], weights["m_hat"], weights["e_hat"])
err_dml = float(torch.max(torch.abs(dml_tau - weights["dml_tau_ref"])))
assert err_dml < 1e-5, f"DML tau parity error: {err_dml:.2e}"

print(f"All parity assertions passed successfully (max error {max(err_anchor, err_y0, err_y1, err_tau, err_mmd, err_prop, err_bce, err_dml):.2e})")


In [ ]:
from core.benchmark import benchmark_fn, format_benchmark_table
b_tarnet = benchmark_fn(lambda: tarnet_forward(weights["test_input"], weights["phi_w"], weights["phi_b"], weights["h0_w"], weights["h0_b"], weights["h1_w"], weights["h1_b"]), iters=100, name="TARNet Dual-Head Forward")
b_mmd = benchmark_fn(lambda: cfrnet_mmd_loss(phi, weights["t_treatment"]), iters=100, name="CFRNet Linear MMD Balance Penalty")
b_dragon = benchmark_fn(lambda: dragonnet_propensity_loss(phi, weights["t_treatment"], weights["w_prop"], weights["b_prop"]), iters=100, name="Dragonnet Propensity Head & BCE")
b_dml = benchmark_fn(lambda: dml_cate_estimate(weights["y_outcome"], weights["t_treatment"], weights["m_hat"], weights["e_hat"]), iters=100, name="Double Machine Learning Robinson CATE")
print(format_benchmark_table([b_tarnet, b_mmd, b_dragon, b_dml]))


In [ ]:
# Failure Mode Contrast: Naive Plug-in Difference vs. Double Machine Learning (DML) Orthogonal Residualization
# Setting: Confounded data where True CATE tau = 2.00
torch.manual_seed(42)
n_sim = 1000
x_sim = torch.randn(n_sim, 2)
# Propensity score with selection bias
e_true = torch.sigmoid(0.8 * x_sim[:, 0] - 0.5 * x_sim[:, 1])
t_sim = torch.bernoulli(e_true).unsqueeze(-1)
# Complex non-linear confounding baseline g(x)
g_x = (x_sim[:, 0] ** 2 + torch.sin(x_sim[:, 1])).unsqueeze(-1)
# Outcome: Y = tau * T + g(X) + noise
y_sim = 2.00 * t_sim + g_x + torch.randn(n_sim, 1) * 0.1

# Naive Plug-in Difference of Means (Unadjusted)
y_treated_mean = y_sim[t_sim == 1.0].mean().item()
y_control_mean = y_sim[t_sim == 0.0].mean().item()
tau_naive = y_treated_mean - y_control_mean

# Double Machine Learning (DML) with Cross-Residualization
# Imperfect nuisance estimates m_hat and e_hat
m_hat_sim = g_x + 2.00 * e_true.unsqueeze(-1) + torch.randn(n_sim, 1) * 0.2
e_hat_sim = e_true.unsqueeze(-1)
tau_dml_sim = dml_cate_estimate(y_sim, t_sim, m_hat_sim, e_hat_sim).item()

print(f"Ground Truth CATE:          2.0000")
print(f"Naive Difference of Means:  {tau_naive:.4f} (Biased by covariate confounding selection!)")
print(f"DML Orthogonal Estimate:    {tau_dml_sim:.4f} (Consistent, recovers true treatment effect!)")

assert abs(tau_naive - 2.00) > 0.08, f"Naive estimate should suffer from selection confounding"
assert abs(tau_dml_sim - 2.00) < 0.05, f"DML estimate should recover true treatment effect"

print("Pathological failure mode contrast successfully verified")


## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Architectural Decoupling**: TARNet dual outcome heads prevent treatment effects from being swallowed by high-dimensional covariate representations.
2. **Balancing vs. Over-Regularization**: CFRNet MMD penalties balance latent representations, but extreme penalties can inadvertently discard predictive covariates. Dragonnet solves this via targeted propensity sufficiency.
3. **Neyman Orthogonal Protection**: DML ensures treatment effect estimates are robust to slow convergence rates in nuisance machine learning estimators.

---
### Cross-Domain Generalisation Matrix
- **Precision Oncology**: TARNet predicts individual drug synergy responses across patient genetic profiles.
- **E-Commerce Dynamic Pricing**: DML isolates price elasticity curves without confounding from seasonal marketing campaigns.
- **Ad-Tech Incrementality**: Separates organic user conversion propensity from true advertisement impact.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [02_causal_treatment_effects_tarnet_cfrnet_dragonnet_dml_and_r_learner.md](../../markdown/09_causal_ai_and_evolutionary_search/02_causal_treatment_effects_tarnet_cfrnet_dragonnet_dml_and_r_learner.md)
- **PyTorch 2.x Module:** [02_causal_treatment_effects_tarnet_cfrnet_dragonnet_dml_and_r_learner.py](../../pytorch2/09_causal_ai_and_evolutionary_search/02_causal_treatment_effects_tarnet_cfrnet_dragonnet_dml_and_r_learner.py)
- **Functional JAX Module:** [02_causal_treatment_effects_tarnet_cfrnet_dragonnet_dml_and_r_learner.py](../../jax/09_causal_ai_and_evolutionary_search/02_causal_treatment_effects_tarnet_cfrnet_dragonnet_dml_and_r_learner.py)
- **Native Rust Candle Track:** [02_causal_treatment_effects_tarnet_cfrnet_dragonnet_dml_and_r_learner.rs](../../candle/09_causal_ai_and_evolutionary_search/02_causal_treatment_effects_tarnet_cfrnet_dragonnet_dml_and_r_learner.rs)
- **Hardware Kernel (CPU / WGSL):** [02_causal_treatment_effects_tarnet_cfrnet_dragonnet_dml_and_r_learner.rs](../../kernels/09_causal_ai_and_evolutionary_search/02_causal_treatment_effects_tarnet_cfrnet_dragonnet_dml_and_r_learner.rs)
- **Golden SafeTensors:** [02_causal_treatment_effects_tarnet_cfrnet_dragonnet_dml_and_r_learner.safetensors](../../weights/09_causal_ai_and_evolutionary_search/02_causal_treatment_effects_tarnet_cfrnet_dragonnet_dml_and_r_learner.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 01. Causal Discovery NOTEARS SCM and Instrumental Variables](01_causal_discovery_notears_scm_and_instrumental_variables.ipynb) | [Master Index](../../README.md#09_causal_ai_and_evolutionary_search) | [Next: 03. Domain Generalization](03_domain_generalization_irm_groupdro_vrex_coral_and_dann.ipynb) &rarr;
